# Experiment 1 — NCF Hyperparameter Tuning (MovieLens 1M)

**STAT 238 · Final Project · UC Berkeley Spring 2026**

---

## Setup

A concurrent CS 289A project compares matrix factorisation, NCF, and a two-stage ranker
on MovieLens 1M to study how recommendation quality degrades as training data sparsity
increases. NCF has five hyperparameters that must be chosen before the sparsity sweep
can be run fairly:

| Parameter | Symbol | Range | Encoding for GP |
|---|---|---|---|
| Embedding dim | $d$ | $\{32,64,128,256\}$ | ordinal $\in[0,3]$ |
| MLP config | — | 3 options | ordinal $\in[0,2]$ |
| Learning rate | $\eta$ | $[10^{-4}, 10^{-2}]$ | $\log\eta \in[-9.21, -4.61]$ |
| L2 weight decay | $\lambda$ | $[10^{-6}, 10^{-3}]$ | $\log\lambda \in[-13.82,-6.91]$ |
| WMF scale | $\alpha$ | $[0.5, 5.0]$ | as-is |

**Objective**: validation NDCG@10 — higher is better.  
**Budget**: 25–30 trials × ~4 min/trial on SCF GPU ≈ 2 hours.  
**Baselines**: random search and grid search at matched budgets.

## Why log-transform $\eta$ and $\lambda$?

The SE kernel assigns distance as $\|x - x'\|^2$.  Without log-transforming,
the difference between $\eta = 10^{-4}$ and $\eta = 10^{-3}$ is $9\times10^{-4}$,
while the difference between $\eta = 10^{-3}$ and $\eta = 10^{-2}$ is $9\times10^{-3}$
— a 10× disparity in Euclidean distance for the same multiplicative change.
In log-space both gaps equal $\log 10 \approx 2.30$, so the GP kernel treats them
equivalently, as it should.

In [ ]:
import sys, math, os
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.bo import BayesianOptimizer
from src.black_box_ncf import (
    NCFBlackBox,
    NCF_PARAM_SPACE,
    decode_params,
    best_ncf_params,
    random_search_ncf,
)

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "figure.dpi": 120})

# -----------------------------------------------------------------------
# Configuration — edit these before running on SCF
# -----------------------------------------------------------------------
CS289_REPO = os.path.expanduser(
    "~/Documents/Berkeley/Spring 2026/CS 289A/Final Project/Github/cs289-ranking"
)
DEVICE     = "cuda"     # 'cuda' on SCF, 'mps' on M-series Mac, 'cpu' for testing
EPOCHS     = 20         # matches proposal; reduce to 5 for a quick sanity run
N_INIT     = 5          # random initialisations before BO starts
BUDGET     = 28         # total evaluations (N_INIT random + BUDGET-N_INIT BO)
SEED       = 42

DRY_RUN    = False      # set True to test the notebook without a GPU

print(f"Device: {DEVICE}  |  Epochs: {EPOCHS}  |  Budget: {BUDGET}")
print(f"Estimated wall time: {BUDGET * 4 / 60:.1f} hours on SCF A4000")

## 1  Bayesian Optimisation Run

In [ ]:
# ---- Instantiate the black box ----
black_box = NCFBlackBox(
    cs289_repo  = CS289_REPO,
    device      = DEVICE,
    epochs      = EPOCHS,
    dry_run     = DRY_RUN,
    verbose     = True,
)

# ---- Run BO ----
bo = BayesianOptimizer(
    objective   = black_box,
    param_space = NCF_PARAM_SPACE,
    n_init      = N_INIT,
    budget      = BUDGET,
    seed        = SEED,
    verbose     = True,
)

bo_results = bo.run()

In [ ]:
# ---- Best configuration found ----
best_decoded = best_ncf_params(bo)
print("=" * 55)
print(f"Best val NDCG@10 : {bo.best_value:.5f}")
print("Best hyperparameters (decoded):")
for k, v in best_decoded.items():
    print(f"  {k:15s}: {v}")
print("=" * 55)
print("\nThis config will be used for the CS 289A sparsity sweep.")

## 2  Random Search Baseline

Random search samples configurations uniformly at random.  We run the same
number of trials as the BO budget to ensure a fair comparison.

In [ ]:
rs_black_box = NCFBlackBox(
    cs289_repo = CS289_REPO,
    device     = DEVICE,
    epochs     = EPOCHS,
    dry_run    = DRY_RUN,
    verbose    = False,
    results_dir= "../results/ncf_random",
)

rs_ndcg = random_search_ncf(rs_black_box, n_trials=BUDGET, seed=SEED + 1)
rs_best = np.maximum.accumulate(rs_ndcg)
print(f"Random search best NDCG@10: {max(rs_ndcg):.5f}")

## 3  Grid Search Baseline

A coarse $2^5 = 32$-point grid across all five dimensions.
This is the most natural alternative when the budget is small
and the search space is discrete.

In [ ]:
grid_black_box = NCFBlackBox(
    cs289_repo = CS289_REPO,
    device     = DEVICE,
    epochs     = EPOCHS,
    dry_run    = DRY_RUN,
    verbose    = False,
    results_dir= "../results/ncf_grid",
)

# ---- Reduced 2×2×2×2×2 grid (32 points) ----
grid_configs = [
    {"emb_dim_x": emb, "mlp_x": mlp,
     "log_lr": math.log(lr), "log_l2": math.log(l2), "alpha": alpha}
    for emb   in [1.0, 3.0]           # 64, 256
    for mlp   in [0.0, 1.0]           # [128,64], [256,128,64]
    for lr    in [1e-3, 5e-4]
    for l2    in [1e-5, 1e-4]
    for alpha in [1.0, 2.5]
]
grid_ndcg = [grid_black_box(cfg) for cfg in grid_configs[:BUDGET]]   # cap at budget
grid_best = np.maximum.accumulate(grid_ndcg)
print(f"Grid search best NDCG@10: {max(grid_ndcg):.5f}")

## 4  Convergence Curves

Best NDCG@10 observed so far vs trial number for all three methods.
This is the main comparison figure.

In [ ]:
bo_best  = bo_results["best_so_far"].values
trials   = np.arange(1, len(bo_best) + 1)
n_common = min(len(bo_best), len(rs_best), len(grid_best))

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(trials[:n_common], bo_best[:n_common],
        color="tab:blue", lw=2.5, label="Bayesian Optimisation (GP + EI)")
ax.plot(trials[:n_common], rs_best[:n_common],
        color="tab:red",  lw=2.5, ls="--", label="Random Search")
ax.plot(trials[:n_common], grid_best[:n_common],
        color="tab:green",lw=2.5, ls=":",  label="Grid Search ($2^5$)")

ax.axvline(N_INIT, color="gray", lw=1, ls=":", alpha=0.7)
ax.text(N_INIT + 0.3, ax.get_ylim()[0] + 0.002, "BO starts", fontsize=9, color="gray")

ax.set_xlabel("Trial number")
ax.set_ylabel("Best val NDCG@10 observed so far")
ax.set_title(
    "NCF hyperparameter tuning — MovieLens 1M\n"
    "Convergence curves: BO vs Random vs Grid Search"
)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
plt.tight_layout()
plt.savefig("../figures/ncf_convergence.png", bbox_inches="tight")
plt.show()

## 5  Trial History

All BO trials, sorted by NDCG@10.

In [ ]:
# Load full trial log with decoded params from CSV
trials_df = pd.read_csv("../results/ncf/trials.csv")

# Sort by NDCG descending
display_df = trials_df.sort_values("val_ndcg", ascending=False).reset_index(drop=True)
display_df.index += 1   # 1-indexed rank

print(f"Total trials: {len(trials_df)}")
print(f"Best NDCG@10: {trials_df['val_ndcg'].max():.5f}")
print(f"Mean NDCG@10: {trials_df['val_ndcg'].mean():.5f}")
print(f"Std  NDCG@10: {trials_df['val_ndcg'].std():.5f}")
print()
print(display_df[["trial","emb_dim","mlp_layers","lr","l2","alpha","val_ndcg","runtime_s"]].head(10).to_string())

In [ ]:
# ---- Distribution of NDCG@10 by BO phase ----
bo_phase  = trials_df[trials_df["trial"] > N_INIT]["val_ndcg"]
rnd_phase = trials_df[trials_df["trial"] <= N_INIT]["val_ndcg"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: NDCG@10 per trial
ax = axes[0]
ax.scatter(trials_df[trials_df["trial"] <= N_INIT]["trial"],
           rnd_phase, c="tab:orange", s=60, label="Random init", zorder=4)
ax.scatter(bo_phase.index + 1, bo_phase,
           c="tab:blue", s=60, label="BO trials", zorder=4)
ax.plot(trials_df["trial"], bo_results["best_so_far"],
        color="black", lw=1.5, ls="--", label="Best so far")
ax.axvline(N_INIT + 0.5, color="gray", lw=1, ls=":")
ax.set_xlabel("Trial"); ax.set_ylabel("val NDCG@10")
ax.set_title("NDCG@10 per trial")
ax.legend(fontsize=9)

# Histogram
ax2 = axes[1]
ax2.hist(rnd_phase, bins=8, alpha=0.6, color="tab:orange", label="Random init")
ax2.hist(bo_phase,  bins=8, alpha=0.6, color="tab:blue",   label="BO trials")
ax2.set_xlabel("val NDCG@10"); ax2.set_ylabel("Count")
ax2.set_title("Distribution of NDCG@10 by phase")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../figures/ncf_trial_scatter.png", bbox_inches="tight")
plt.show()

## 6  GP Surrogate — Final State

Visualise the GP posterior mean over the two most impactful dimensions
($\log\eta$ and $\log\lambda$), holding emb_dim, mlp_layers, and $\alpha$
fixed at the best found values.

In [ ]:
from src.gp import GaussianProcess
from src.acquisition import expected_improvement

# Rebuild the GP on all BO observations in normalised space
lo = np.array([p["bounds"][0] for p in NCF_PARAM_SPACE])
hi = np.array([p["bounds"][1] for p in NCF_PARAM_SPACE])

X_all_raw = bo_results[[p["name"] for p in NCF_PARAM_SPACE]].values   # (n, 5)
y_all     = bo_results["objective"].values

X_unit = (X_all_raw - lo) / (hi - lo)
y_std  = (y_all - y_all.mean()) / y_all.std()

gp_final = GaussianProcess(n_restarts=5)
gp_final.condition(X_unit, y_std)
print(f"Final GP hyperparams: {gp_final}")

# ---- 2D slice: log_lr vs log_alpha (dims 2 and 4), fix others at best ----
best_enc = bo.best_params
best_unit = (np.array([best_enc[p["name"]] for p in NCF_PARAM_SPACE]) - lo) / (hi - lo)

n_grid = 60
lr_range  = np.linspace(0, 1, n_grid)   # dim 2 (log_lr)
l2_range  = np.linspace(0, 1, n_grid)   # dim 3 (log_l2)
LR, L2    = np.meshgrid(lr_range, l2_range)

# Build grid: hold all dims at best_unit, sweep dims 2 and 3
base = np.tile(best_unit, (n_grid * n_grid, 1))
base[:, 2] = LR.ravel()
base[:, 3] = L2.ravel()

mu_grid, var_grid = gp_final.predict(base)
ei_grid = expected_improvement(base, gp_final, y_best=y_std.max())

# Map axes back to original scale for labelling
log_lr_axis = lo[2] + lr_range * (hi[2] - lo[2])
log_l2_axis = lo[3] + l2_range * (hi[3] - lo[3])
LR_orig, L2_orig = np.meshgrid(np.exp(log_lr_axis), np.exp(log_l2_axis))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, Z, label in [
    (axes[0], mu_grid.reshape(n_grid, n_grid), r"Posterior mean $\mu(x)$ (standardised)"),
    (axes[1], ei_grid.reshape(n_grid, n_grid), r"EI$(x)$"),
]:
    ct = ax.contourf(LR_orig, L2_orig, Z, levels=30, cmap="viridis")
    fig.colorbar(ct, ax=ax)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"Learning rate $\eta$")
    ax.set_ylabel(r"L2 weight decay $\lambda$")
    ax.set_title(label)
    # Mark best
    best_lr = np.exp(best_enc["log_lr"])
    best_l2 = np.exp(best_enc["log_l2"])
    ax.plot(best_lr, best_l2, "r*", markersize=14, label="Best config")
    ax.legend(fontsize=9)

plt.suptitle(
    "GP surrogate — final state after all BO trials\n"
    r"Slice: $\eta$ vs $\lambda$ (other dims fixed at best)",
    y=1.02
)
plt.tight_layout()
plt.savefig("../figures/ncf_gp_final.png", bbox_inches="tight")
plt.show()

## 7  Best Configuration — CS 289A Handoff

The best config found by BO is used directly in the CS 289A sparsity sweep.
Copy the command below and run it on SCF at each density level.

In [ ]:
best = best_ncf_params(bo)

print("=" * 60)
print("Best NCF configuration found by BO")
print("=" * 60)
print(f"  emb_dim    : {best['emb_dim']}")
print(f"  mlp_layers : {best['mlp_layers']}")
print(f"  lr         : {best['lr']:.2e}")
print(f"  l2         : {best['l2']:.2e}")
print(f"  alpha      : {best['alpha']:.4f}")
print(f"  val NDCG@10: {bo.best_value:.5f}")
print()

mlp_str = " ".join(str(h) for h in best['mlp_layers'])
for density in [1.0, 0.8, 0.6, 0.4, 0.2]:
    cmd = (
        f"python src/train.py "
        f"--model ncf --density {density} "
        f"--emb-dim {best['emb_dim']} "
        f"--mlp-layers {mlp_str} "
        f"--lr {best['lr']:.2e} "
        f"--l2 {best['l2']:.2e} "
        f"--alpha {best['alpha']:.4f} "
        f"--epochs 20 --device cuda"
    )
    print(f"# density = {density}")
    print(cmd)
    print()

## 8  Summary

| Metric | BO | Random Search | Grid Search |
|---|---|---|---|
| Best val NDCG@10 | — | — | — |
| Trials to reach 0.48+ | — | — | — |
| Total evaluations | 28 | 28 | 28 |
| Total wall time | ~2h | ~2h | ~2h |

*(fill in after the run)*

---

**References**
- Guntuboyina (2026). STAT 238 Lecture 22–23. UC Berkeley.  
- Frazier (2018). A Tutorial on Bayesian Optimization. *arXiv:1807.02811*.  
- Snoek, Larochelle, Adams (2012). Practical BO of ML Algorithms. *NeurIPS 2012*.  
- Harper & Konstan (2015). The MovieLens Datasets. *ACM TIIS*.